## Topic: Wikipedia Retriever

### Agenda
- 1. Introduction of Wikipedia Retriever

- 2. How to Work internally?

- 3. Syntax of Wikipedia Retriever and when to use?

- 4. Practical Examples of Wikipedia Retriever 

- 5. Complete Summary of Wikipedia Retriever

### 1. Introduction of Wikipedia Retriever

- Definition:
    - A Wikipedia Retriever is a retriever that queries the Wikipedia API to fetch relevant content for a given query.


    - Wikipedia Retriever is an external API retriever in LangChain that fetches relevant documents live from Wikipedia instead of from our own vector store.

- Key Note:
    - No embeddings. No index building. No ingestion pipeline. Just query -> Wikipedia API -> List[Document].

In [ ]:
"""
    - Workflow
    ===========

Query: "What is Photosynthesis?"
            ↓
  [WikipediaRetriever]
            ↓ calls Wikipedia API search
  Top-3 Wikipedia pages + content
            ↓
  [Document(page_content="...", metadata={"title":..., "source":...})]

"""

### 2. How to Work internally?

- How it Work?
    - 1. Receive Query 
    - 2. It send to the query to Wikipedia's API
    - 3. It retrieves the most relevant articles
    - 4. It returns them as LangChain Document objects.

In [ ]:
""" 

        WIKIPEDIA RETRIEVER INTERNAL FLOW 
        =====================================

    ┌─────────────────────────────────────────────────────────────┐
    │            WIKIPEDIA RETRIEVER INTERNAL FLOW                │
    │                                                             │
    │  1. RECEIVE QUERY                                           │
    │     retriever.invoke("Large Language Models")               │
    │                                                             │
    │  2. CALL WIKIPEDIA SEARCH API                               │
    │     wikipedia.search(query, results=top_k_results)          │
    │     → Returns page TITLES ranked by Wikipedia:              │
    │       ["Large language model", "Generative AI", "GPT-4"]    │
    │                                                             │
    │  3. FETCH PAGE CONTENT (per title)                          │
    │     For each title:                                         │
    │       page = wikipedia.page(title, auto_suggest=False)      │
    │       content = page.content[:doc_content_chars_max]        │
    │                                                             │
    │     Handles: DisambiguationError, PageError, Redirects      │
    │                                                             │
    │  4. BUILD DOCUMENT OBJECTS                                  │
    │     Document(                                               │
    │         page_content="Large language models (LLMs) are...", │
    │         metadata={                                          │
    │             "title": "Large language model",                │
    │             "source": "https://en.wikipedia.org/wiki/...",  │
    │             "summary": "First paragraph summary..."         │
    │         }                                                   │
    │     )                                                       │
    │                                                             │
    │  5. RETURN List[Document]                                   │
    │     [Document(...), Document(...), Document(...)]           │
    └─────────────────────────────────────────────────────────────┘

"""

### 3. Syntax of Wikipedia Retriever and when to use?

In [ ]:
""" 
    - 1. Required packages
    =========================

        - pip install langchain langchain-community wikipedia

"""


""" 
    - 2. Basic Syntax
    =========================

    from langchain_community.retrievers import WikipediaRetriever

    retriever = WikipediaRetriever(
        top_k_results=3,          # how many pages to return
        lang="en",                # Wikipedia return language
        load_all_available_meta=True,  # richer metadata
        doc_content_chars_max=4000,    # truncate very long articles(token control)
    )

    docs = retriever.invoke("Quantum computing")



"""

### Another Approach

""" 
WikipediaRetriever(
    top_k_results: int = 3,            # How many pages to return
    lang: str = "en",                  # "en", "fr", "de", "es", "hi", etc.
    doc_content_chars_max: int = 4000, # Truncate each page to N chars
    load_max_docs: int = 100,          # Internal search cap (rarely change)
)


"""


""" 
 # RETRIEVER: Search by query → returns docs (dynamic)
 ----------------------------------------------------------

    from langchain_community.retrievers import WikipediaRetriever
    retriever = WikipediaRetriever(top_k_results=2)
    docs = retriever.invoke("quantum computing")  # ← free-text search

# LOADER: Load SPECIFIC pages by exact title (static)
----------------------------------------------------------


    from langchain_community.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query="Quantum computing", load_max_docs=2)
    docs = loader.load()  # ← loads exact pages



- Key Rule of thumb:
    -  Use Retriever when query is unknown at build-time (chatbot). Use Loader when you know exactly which pages to ingest into FAISS.


"""

- Use it when:
    - You need public, factual, encyclopedic knowledge
    - The user asks about people, history, science, geography, famous products
    - You do not have (or do not want) a local Wikipedia dump
    - You want citations (metadata["source"] URLs)
    
- Do not use it as the only retriever when:
    - Data is private (policies, tickets, internal PDFs)
    - You need exact product SKUs / error codes from your systems
    - You need strict offline operation
    - Wikipedia is too broad / noisy for a narrow domain (use your own FAISS corpus)

### 4. Practical Examples of Wikipedia Retriever 

In [3]:
# import WikipediaRetriever
from langchain_community.retrievers import WikipediaRetriever

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


In [4]:
# initialize the retriever 
retriever = WikipediaRetriever(
    top_k_results = 3,
    lang = "en"
)

In [9]:
# Define Query
query = "Who is the Current President of bangladesh?"

# Get the relevant Wikipedia documents
docs = retriever.invoke(query)

In [10]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n----Result{i + 1} ----")
    print(f"Content: \n{doc.page_content}")


----Result1 ----
Content: 
The President of Bangladesh (POB), officially the President of the People's Republic of Bangladesh, is the head of state of Bangladesh and the commander-in-chief of the Bangladesh Armed Forces. While the role has been ceremonial since the 1990 uprising, the president has gained greater political influence since the 2026 presidential election.
The president is elected by the Jatiya Sangsad for a five-year term and represents the nation in domestic and international affairs.
The role of the president has changed three times since Bangladesh achieved its independence in 1971. Initially, during the first 20 years the president held executive power. In 1991, with the restoration of a democratically elected government, Bangladesh adopted a parliamentary democracy based on a Westminster system, making the president largely ceremonial. 
In 1996, the constitution passed new laws slightly enhancing the president's executive authority only after a Parliament dissolutio

In [ ]:
# # Define Query
# query = "the geopolitical history of india and bangladesh from the perspective of USA"

# # Get the relevant Wikipedia documents
# docs = retriever.invoke(query)

In [ ]:
"""
         Wikipedia Retriever vs Document loader
        ================================================

- Core Difference
       -  Document Loader = brings documents into LangChain.
       -  Wikipedia Retriever = searches Wikipedia and retrieves relevant documents for a query.

"""

### 5. Complete Summary of Wikipedia Retriever

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                    WIKIPEDIA RETRIEVER                           │
│                                                                  │
│  WHAT:  Retrieves relevant Wikipedia pages as Documents          │
│  WHY:   Ground LLM answers in public encyclopedic knowledge      │
│  WHERE: Query-time external source OR ingest step before FAISS   │
│                                                                  │
│  INTERNAL FLOW:                                                  │
│    query → Wikipedia search → fetch pages → Document objects     │
│                                                                  │
│  SYNTAX:                                                         │
│    from langchain_community.retrievers import WikipediaRetriever │
│    wiki = WikipediaRetriever(                                    │
│        top_k_results=3,                                          │
│        lang="en",                                                │
│        doc_content_chars_max=4000                                │
│    )                                                             │
│    docs = wiki.invoke("Quantum computing")                       │
│                                                                  │
│  KEY PARAMS:                                                     │
│    top_k_results         → number of pages                       │
│    lang                  → Wikipedia language                    │
│    doc_content_chars_max → truncate long articles                │
│    load_all_available_meta → richer metadata                     │
│                                                                  │
│  WHEN TO USE:                                                    │
│    ✅ Facts, people, history, science, general knowledge          │
│    ✅ Need citations / URLs                                       │
│    ✅ No local Wikipedia dump                                     │
│    ❌ Private company data                                        │
│    ❌ Offline-only systems                                        │
│                                                                  │
│  WITH FAISS:                                                     │
│    WikipediaRetriever.invoke(q)                                  │
│         → split → embed → FAISS.from_documents()                 │
│         → vectorstore.as_retriever(search_type="similarity")     │
│                                                                  │
│  PIPELINE OPTIONS:                                               │
│    A) Query → Wikipedia → LLM                                    │
│    B) Wikipedia ingest → FAISS → Similarity retriever → LLM      │
│    C) Ensemble(FAISS private + Wikipedia public) → LLM           │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Use Wikipedia for public facts. Index it in FAISS if you       │
│   will query the same topics repeatedly. Never use it as a       │
│   substitute for your private knowledge base."                   │
└──────────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                   WIKIPEDIA RETRIEVER                            │
│                                                                  │
│  WHAT:  Live retriever that searches Wikipedia.org and returns   │
│         top pages as LangChain Documents (no embeddings needed)  │
│                                                                  │
│  HOW:   query → Wikipedia search API → fetch pages →             │
│         truncate → return List[Document]                         │
│                                                                  │
│  SYNTAX:                                                         │
│    from langchain_community.retrievers import WikipediaRetriever │
│    retriever = WikipediaRetriever(                               │
│        top_k_results=3,            # pages to fetch              │
│        doc_content_chars_max=2000, # chars per page              │
│        lang="en"                    # language edition           │
│    )                                                             │
│    docs = retriever.invoke("query")                              │
│                                                                  │
│  KEY PARAMS:                                                     │
│    top_k_results         → Pages returned (default: 3)           │
│    doc_content_chars_max → Truncation per page (default: 4000)   │
│    lang                  → en, fr, de, es, ja, etc.              │
│                                                                  │
│  METADATA RETURNED:                                              │
│    title   → Wikipedia page title                                │
│    source  → Full Wikipedia URL                                  │
│    summary → Lead paragraph                                      │
│                                                                  │
│  WITH FAISS (RECOMMENDED PATTERN):                               │
│    WikipediaRetriever → fetch once → split chunks →              │
│    FAISS.from_documents() → save_local() → offline semantic RAG  │
│                                                                  │
│  STRENGTHS:                                                      │
│    ✅ Zero setup, no embedding cost                               │
│    ✅ Always live / up-to-date                                   │
│    ✅ Free, multilingual (lang parameter)                        │
│    ✅ Same invoke() interface → plugs into any RAG chain        │
│    ✅ Great fallback when local FAISS misses                     │
│                                                                  │
│  WEAKNESSES:                                                     │
│    ❌ Keyword search only (not semantic)                         │
│    ❌ Slow (~1-3s per query, network-dependent)                  │
│    ❌ No private data                                            │
│    ❌ Long pages → must truncate / chunk                         │
│    ❌ Rate limits, PageErrors, non-deterministic                 │
│                                                                  │
│  WHEN TO USE vs ALTERNATIVES:                                    │
│    WikipediaRetriever      → General knowledge, prototyping      │
│    Wikipedia → FAISS       → Offline, semantic, production       │
│    Ensemble (FAISS+Wiki)   → Private + public hybrid             │
│    Tavily / Web search     → Recent news, beyond Wikipedia       │
│    ArxivRetriever          → Academic / research papers          │
│                                                                  │
│  PIPELINE:                                                       │
│    DIRECT:  Query → [Wikipedia API] → Docs → [LLM]               │
│                         THIS COMPONENT                           │
│    CACHED:  Wiki → [Splitter] → [Embed] → [FAISS] → [Retriever]  │
│                                                   → [LLM]        │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Use WikipediaRetriever for prototyping and fallback.           │
│   For production, ingest Wikipedia pages into FAISS ONCE,        │
│   then serve with semantic search — faster, cheaper, offline."   │
└──────────────────────────────────────────────────────────────────┘
"""